# 20.12 助理答錯時，怎麼分清聽錯與回錯？


你說「請推薦不辣的晚餐」，辨識器漏掉「不」，助理就收到相反條件。聊天模型可能很認真推薦辣菜，卻沒有回答你的原需求。要修好這個系統，先找出紙條在哪一棒被改了。

前置是[12.13的聽寫與回答](https://birdhackor.github.io/tiny-perceptron-vlm/12.13.html)及[20.1的共同入口](https://birdhackor.github.io/tiny-perceptron-vlm/20.1.html)。ASR的任務是寫出聽到的話，聊天的任務是依收到的話回應。把正確逐字稿直接送進聊天，可以檢查第二站；把真ASR結果送進相同聊天，才是完整語音路線。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/natural-v4-asr-two-routes.svg")))

先用手寫漏字例子看影響：


In [ ]:
from tiny_perceptron.natural_concepts import text_error_report

reference = "請推薦不辣的晚餐。"
recognized = "請推薦辣的晚餐。"
report = text_error_report(reference, recognized)
print("最少編輯次數", report["edits"])
print("參考字數", report["reference_characters"])
print("CER百分比", round(report["cer"] * 100, 1))
print("兩份問題相同", reference == recognized)

只漏一個字，輸出1、9、11.1與False。這個CER描述轉寫的字元差距，不能代替需求是否仍相同：一個否定詞就能反轉限制。程式沒有播放錄音，也沒有請聊天模型生成；它隔離的是中間文字改動。

正式檢查留下三項：ASR逐字稿與來源原話的差距，正確文字輸入時的聊天回應，以及真ASR輸入時的聊天回應。兩路使用相同歷史、問題要求與照片。若正確文字已答不好，問題不全在ASR；若文字能答、辨識後不能答，再查看關鍵字是否被聽錯。

本課用真人朗讀的問句檢查這條路，例如詢問如何加強溝通。來源給錄音與逐字稿；合理助手回應的判準由本課在推論前另訂。這些是人讀出來的問句，不是自然錄下的即興聊天，仍需另外測口音、噪聲、多人重疊與長對話。ASR採現成權重時，這份檢查也不是本課重新訓練辨識器的證據。

使用者可以在介面改正逐字稿，再送出。這是明確的人工操作；評估原始語音路線時，仍保存更正前辨識，不偷偷換成來源真值。練習把「不」補回，確認CER變0，再想：即使聽寫完全正確，聊天還要滿足哪些條件？完整能力卡會把兩站的結果分開回答。
